In [ ]:
!git clone https://github.com/RIA-lab/MMTD.git

In [ ]:
%cd MMTD

In [ ]:
!pip install -r requirements.txt

In [ ]:
!pip install transformers datasets torch torchvision torchaudio opencv-python pillow

In [ ]:
%cat models.py

In [ ]:
%%writefile -a models.py


class PhishGuard_MMTD(torch.nn.Module):
    def __init__(self, bert_cfg=BertConfig(), beit_cfg=BeitConfig(), bert_pretrain_weight=None, beit_pretrain_weight=None):
        super(PhishGuard_MMTD, self).__init__()
        
        # 1. Temel Modeller
        self.text_encoder = BertForSequenceClassification.from_pretrained(bert_pretrain_weight) if bert_pretrain_weight is not None else BertForSequenceClassification(bert_cfg)
        self.image_encoder = BeitForImageClassification.from_pretrained(beit_pretrain_weight) if beit_pretrain_weight is not None else BeitForImageClassification(beit_cfg)
        
        self.text_encoder.config.output_hidden_states = True
        self.image_encoder.config.output_hidden_states = True
        
        # 2. VRAM TASARRUFU: Ağırlıkları Dondur
        for param in self.text_encoder.parameters():
            param.requires_grad = False
        for param in self.image_encoder.parameters():
            param.requires_grad = False

        # 3. CO-ATTENTION KATMANLARI
        self.text_to_image_attn = torch.nn.MultiheadAttention(embed_dim=768, num_heads=8, batch_first=True)
        self.image_to_text_attn = torch.nn.MultiheadAttention(embed_dim=768, num_heads=8, batch_first=True)

        # Classifier şimdilik 768*2 boyutunda, Gating eklenince değişecek
        self.classifier = torch.nn.Linear(768 * 2, 2) 
        self.num_labels = 2
        self.device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

    def forward(self, input_ids, token_type_ids, attention_mask, pixel_values, labels=None):
        # A. Modellerden Vektörleri Al
        text_outputs = self.text_encoder(input_ids=input_ids, token_type_ids=token_type_ids, attention_mask=attention_mask)
        image_outputs = self.image_encoder(pixel_values=pixel_values)
        
        text_hidden = text_outputs.hidden_states[12]
        image_hidden = image_outputs.hidden_states[12]
        
        # B. CO-ATTENTION SİHRİ
        text_attended, _ = self.text_to_image_attn(query=text_hidden, key=image_hidden, value=image_hidden)
        image_attended, _ = self.image_to_text_attn(query=image_hidden, key=text_hidden, value=text_hidden)
        
        # C. HAVUZLAMA (Sadece CLS tokeni)
        text_pooled = text_attended[:, 0, :]
        image_pooled = image_attended[:, 0, :]
        
        # D. ŞİMDİLİK BASİT FÜZYON
        fused_features = torch.cat([text_pooled, image_pooled], dim=1)
        
        logits = self.classifier(fused_features)
        
        loss = None
        if labels is not None:
            loss_fct = CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
            
        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=None,
            attentions=None,
        )

In [ ]:
# 1. Klasörde olduğumuzdan emin olalım
%cd /kaggle/working/MMTD

import torch
from models import PhishGuard_MMTD

print("1. Model ayağa kaldırılıyor... (Ağırlıklar internetten çekilecek, bekleyiniz...)")
# Cihazı seç (Kaggle'da GPU açıksa CUDA, kapalıysa CPU kullanır)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Kullanılan donanım: {device}")

# Modeli oluştur ve cihaza gönder
model = PhishGuard_MMTD().to(device)
model.eval() # Modeli test (inference) moduna alıyoruz

print("2. Sahte (Dummy) Veriler Üretiliyor...")
batch_size = 2 # Sisteme aynı anda 2 e-posta veriyormuşuz gibi düşün

# A) Sahte Metin Verisi (2 e-posta, 512 kelime uzunluğunda)
dummy_input_ids = torch.randint(0, 30000, (batch_size, 512)).to(device)
dummy_attention_mask = torch.ones(batch_size, 512).to(device)
dummy_token_type_ids = torch.zeros(batch_size, 512, dtype=torch.long).to(device)

# B) Sahte Görsel Verisi (2 e-posta ekran görüntüsü, 3 renk kanalı, 224x224 piksel)
dummy_pixel_values = torch.randn(batch_size, 3, 224, 224).to(device)

print("3. Co-Attention Testi Başlıyor... Çarpışma Testi (Forward Pass)")

try:
    with torch.no_grad(): # Test yaptığımız için hafızayı (RAM/VRAM) yormuyoruz
        outputs = model(
            input_ids=dummy_input_ids,
            token_type_ids=dummy_token_type_ids,
            attention_mask=dummy_attention_mask,
            pixel_values=dummy_pixel_values
        )
    
    print("\n✅ MÜKEMMEL SONUÇ! Co-Attention katmanı kusursuz çalışıyor.")
    print(f"Çıkan Karar Matrisi Boyutu: {outputs.logits.shape}")
    print("Beklenen boyut [2, 2] olmalıydı. (Yani 2 e-posta için Phishing/Benign ihtimalleri)")
    
except Exception as e:
    print("\n❌ HATA! Matris boyutlarında bir çakışma var:")
    print(e)

In [ ]:
%%writefile -a models.py

class PhishGuard_MMTD_Gated(torch.nn.Module):
    def __init__(self, bert_cfg=BertConfig(), beit_cfg=BeitConfig(), bert_pretrain_weight=None, beit_pretrain_weight=None):
        super(PhishGuard_MMTD_Gated, self).__init__()
        
        # 1. Temel Modeller
        self.text_encoder = BertForSequenceClassification.from_pretrained(bert_pretrain_weight) if bert_pretrain_weight is not None else BertForSequenceClassification(bert_cfg)
        self.image_encoder = BeitForImageClassification.from_pretrained(beit_pretrain_weight) if beit_pretrain_weight is not None else BeitForImageClassification(beit_cfg)
        
        self.text_encoder.config.output_hidden_states = True
        self.image_encoder.config.output_hidden_states = True
        
        # 2. VRAM TASARRUFU: Ağırlıkları Dondur
        for param in self.text_encoder.parameters():
            param.requires_grad = False
        for param in self.image_encoder.parameters():
            param.requires_grad = False

        # 3. CO-ATTENTION KATMANLARI
        self.text_to_image_attn = torch.nn.MultiheadAttention(embed_dim=768, num_heads=8, batch_first=True)
        self.image_to_text_attn = torch.nn.MultiheadAttention(embed_dim=768, num_heads=8, batch_first=True)

        # 4. YENİ ZEKAMIZ: Dinamik Gating Ağı (Gate 2 Hakemi)
        # İki 768'lik vektörü alıp 1 tane Alpha (0-1 arası) üretecek
        self.gating_network = torch.nn.Sequential(
            torch.nn.Linear(768 * 2, 256),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.1),
            torch.nn.Linear(256, 1),
            torch.nn.Sigmoid()
        )

        # DİKKAT: Artık Classifier 768*2 boyutunda değil, sadece 768 boyutunda veri alacak!
        # Çünkü alfa ile ağırlıklandırıp topladığımızda matris boyutu genişlemez.
        self.classifier = torch.nn.Linear(768, 2) 
        self.num_labels = 2
        self.device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

    def forward(self, input_ids, token_type_ids, attention_mask, pixel_values, labels=None):
        # A. Vektörleri Al
        text_outputs = self.text_encoder(input_ids=input_ids, token_type_ids=token_type_ids, attention_mask=attention_mask)
        image_outputs = self.image_encoder(pixel_values=pixel_values)
        
        text_hidden = text_outputs.hidden_states[12]
        image_hidden = image_outputs.hidden_states[12]
        
        # B. CO-ATTENTION
        text_attended, _ = self.text_to_image_attn(query=text_hidden, key=image_hidden, value=image_hidden)
        image_attended, _ = self.image_to_text_attn(query=image_hidden, key=text_hidden, value=text_hidden)
        
        text_pooled = text_attended[:, 0, :]
        image_pooled = image_attended[:, 0, :]
        
        # C. DİNAMİK GATING (Gate 2) SİHRİ BAŞLIYOR
        # Hakemin karar verebilmesi için ikisini geçici olarak birleştirip ağa veriyoruz
        concat_for_gate = torch.cat([text_pooled, image_pooled], dim=1)
        
        # Hakem (Ağ) Alpha değerini üretiyor. Boyut: [Batch, 1]
        alpha = self.gating_network(concat_for_gate)
        
        # D. AKILLI FÜZYON (Alpha'ya göre çarpıp topluyoruz)
        # alpha metni, (1-alpha) görseli kontrol ediyor
        fused_features = (alpha * text_pooled) + ((1 - alpha) * image_pooled)
        
        logits = self.classifier(fused_features)
        
        loss = None
        if labels is not None:
            loss_fct = CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
            
        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=None,
            attentions=None,
        )

In [ ]:
import torch
from models import PhishGuard_MMTD_Gated

print("1. Gated Model (Hakemli) ayağa kaldırılıyor...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = PhishGuard_MMTD_Gated().to(device)
model.eval() 

print("2. Sahte (Dummy) Veriler Üretiliyor...")
batch_size = 2 

dummy_input_ids = torch.randint(0, 30000, (batch_size, 512)).to(device)
dummy_attention_mask = torch.ones(batch_size, 512).to(device)
dummy_token_type_ids = torch.zeros(batch_size, 512, dtype=torch.long).to(device)
dummy_pixel_values = torch.randn(batch_size, 3, 224, 224).to(device)

print("3. Dinamik Gating Testi Başlıyor... (Forward Pass)")

try:
    with torch.no_grad():
        outputs = model(
            input_ids=dummy_input_ids,
            token_type_ids=dummy_token_type_ids,
            attention_mask=dummy_attention_mask,
            pixel_values=dummy_pixel_values
        )
    
    print("\n✅ KUSURSUZ! Dinamik Gating (Gate 2) hatasız çalışıyor.")
    print(f"Çıkan Karar Matrisi Boyutu: {outputs.logits.shape}")
    print("Beklenen boyut [2, 2]. Artık beyni ana gövdeye bağlamaya hazırız!")
    
except Exception as e:
    print("\n❌ HATA! Gating matematiğinde bir çakışma var:")
    print(e)

In [ ]:
%cat main.py


In [ ]:
%%writefile main.py
from Email_dataset import EDPDataset, EDPCollator
# 1. BİZİM HAKEMLİ MODELİMİZİ ÇAĞIRIYORUZ
from models import PhishGuard_MMTD_Gated 
from transformers import Trainer, TrainingArguments
from torch.optim import AdamW
from utils import metrics, SplitData, save_config, EvalMetrics
import wandb
import os

fold = 5
split_data = SplitData('DATA/email_data/EDP.csv', fold)

# 2. DOSYA YOLU GÜVENLİĞİ (Çökme Engelleyici)
bert_checkpoint_path = './output/bert/checkpoints'
dit_checkpoint_path = './output/dit/checkpoints'

if __name__ == '__main__':
    for i in range(fold):
        wandb.init(project='PhishGuard-MMTD')
        wandb.run.name = 'PhishGuard-fold-' + str(i + 1)
        train_df, test_df = split_data()
        train_dataset = EDPDataset('DATA/email_data/pics', train_df)
        test_dataset = EDPDataset('DATA/email_data/pics', test_df)

        # Önceden eğitilmiş ağırlıklar varsa al, yoksa HuggingFace'ten sıfır indir
        bert_checkpoint = None
        dit_checkpoint = None
        if os.path.exists(bert_checkpoint_path) and len(os.listdir(bert_checkpoint_path)) > i:
            bert_folds = os.listdir(bert_checkpoint_path)
            bert_checkpoints = os.listdir(os.path.join(bert_checkpoint_path, bert_folds[i]))
            bert_checkpoint = os.path.join(bert_checkpoint_path, bert_folds[i], bert_checkpoints[0])
            
        if os.path.exists(dit_checkpoint_path) and len(os.listdir(dit_checkpoint_path)) > i:
            dit_folds = os.listdir(dit_checkpoint_path)
            dit_checkpoints = os.listdir(os.path.join(dit_checkpoint_path, dit_folds[i]))
            dit_checkpoint = os.path.join(dit_checkpoint_path, dit_folds[i], dit_checkpoints[0])

        # 3. YENİ BEYNİ SİSTEME TAKIYORUZ
        model = PhishGuard_MMTD_Gated(bert_pretrain_weight=bert_checkpoint, beit_pretrain_weight=dit_checkpoint)

        # Ağırlıkların dondurulması işlemi artık models.py içindeki __init__ içinde yapıldığı için
        # buradaki dondurma döngülerine gerek kalmadı ama garanti olsun diye bırakıyoruz.
        filtered_parameters = []
        for p in filter(lambda p: p.requires_grad, model.parameters()):
            filtered_parameters.append(p)

        optimizer = AdamW(filtered_parameters, lr=5e-4)

        args = TrainingArguments(
            output_dir='./output/PhishGuard/checkpoints/fold' + str(i + 1),
            logging_dir='./output/PhishGuard/log',
            logging_strategy='epoch',
            learning_rate=5e-4,
            per_device_train_batch_size=8, # 4. VRAM KURTARICISI (40'tan 8'e düşürdük)
            per_device_eval_batch_size=8,
            num_train_epochs=3,
            weight_decay=0.0,
            save_strategy="epoch",
            evaluation_strategy="epoch",
            load_best_model_at_end=True,
            dataloader_num_workers=2, # Veri yüklemeyi biraz hızlandırır
            dataloader_pin_memory=True,
            run_name=wandb.run.name,
            auto_find_batch_size=False,
            overwrite_output_dir=True,
            save_total_limit=2, # Kaggle diskini doldurmamak için 5'ten 2'ye çektik
            remove_unused_columns=False,
            report_to=["wandb"],
        )

        trainer = Trainer(
            model=model,
            args=args,
            train_dataset=train_dataset,
            eval_dataset=test_dataset,
            optimizers=(optimizer, None),
            data_collator=EDPCollator(),
            compute_metrics=metrics,
        )

        # 5. İKİNCİ GEREKSİZ TRAIN SİLİNDİ
        trainer.train()

        train_acc = trainer.evaluate(eval_dataset=train_dataset)
        train_result = {'train_acc': train_acc['eval_acc'], 'train_loss': train_acc['eval_loss']}
        wandb.log(train_result)

        trainer.compute_metrics = EvalMetrics('./output/PhishGuard/results', wandb.run.name, True)
        test_acc = trainer.evaluate(eval_dataset=test_dataset)
        test_result = {'test_acc': test_acc['eval_acc'], 'test_loss': test_acc['eval_loss']}
        wandb.log(test_result)

        wandb.config = args.to_dict()
        save_config(args.to_dict(), os.path.join('./output/PhishGuard/configs', wandb.run.name + '.yaml'))
        wandb.finish()

In [ ]:
%cd /kaggle/working/MMTD/DATA


In [ ]:
!pip install -U gdown

In [ ]:
!gdown --id "1w4HxNf099lQ41mhMyXzbGwI429qBGvbY"

In [ ]:
# DATA klasörünün içindeki tüm zip dosyalarını buraya çıkar
!unzip -q *.zip

# Çıkan dosyaların doğru yerde olup olmadığını kontrol edelim
!ls -la

In [ ]:
# 1. DATA klasörüne git ve o devasa ZIP dosyasını sil (6GB alan aç)
%cd /kaggle/working/MMTD/DATA
!rm email_data.zip
print("✅ Temizlik tamamlandı! Disk rahat nefes aldı.")

# 2. Ana dizine dön ve MOTORU ATEŞLE!
%cd /kaggle/working/MMTD
!python main.py

In [ ]:
# 1. Klasörde olduğumuzdan emin olalım
%cd /kaggle/working/MMTD

# 2. NEŞTER MÜDAHALESİ: Email_dataset.py içindeki eski ismi yeni isimle saniyeler içinde değiştiriyoruz (Linux sed komutu ile)
!sed -i 's/BeitFeatureExtractor/BeitImageProcessor/g' Email_dataset.py

# 3. WANDB TUZAĞINI İPTAL ET VE MOTORU ATEŞLE
# Wandb'yi offline moda alıp şifre sormasını engelliyoruz ve eğitimi başlatıyoruz
!export WANDB_MODE=offline && python main.py

In [ ]:
%cd /kaggle/working/MMTD

# 1. Kalan tüm eski FeatureExtractor'ları yeni ImageProcessor isimleriyle topluca değiştiriyoruz
!sed -i 's/CLIPFeatureExtractor/CLIPImageProcessor/g' Email_dataset.py
!sed -i 's/ViltFeatureExtractor/ViltImageProcessor/g' Email_dataset.py

# 2. WandB'yi susturup sistemi tekrar ayağa kaldırıyoruz
!export WANDB_MODE=offline && python main.py

In [ ]:
%cd /kaggle/working/MMTD

# Bütün Python dosyalarındaki sinsi eski kütüphane isimlerini tek hamlede güncelliyoruz:
!sed -i 's/CLIPFeatureExtractor/CLIPImageProcessor/g' *.py
!sed -i 's/ViltFeatureExtractor/ViltImageProcessor/g' *.py
!sed -i 's/BeitFeatureExtractor/BeitImageProcessor/g' *.py

print("✅ Tüm projedeki eski bağımlılıklar temizlendi!")

# WANDB'yi kapatıp motoru son kez ateşliyoruz
!export WANDB_MODE=offline && python main.py

In [ ]:
%cd /kaggle/working/MMTD

# 1. Eksik olan torchtext kütüphanesini hızlıca kuruyoruz
!pip install torchtext

print("✅ Torchtext kuruldu! Artık utils.py şikayet etmeyecek.")

# 2. Motoru (Umarım bu sefer gerçekten) ateşliyoruz!
!export WANDB_MODE=offline && python main.py

In [ ]:
%cd /kaggle/working/MMTD

# utils.py dosyasının içindeki içinde 'torchtext' veya 'GloVe' geçen tüm satırları acımadan siliyoruz!
!sed -i '/torchtext/d' utils.py
!sed -i '/GloVe/d' utils.py

print("✅ Gereksiz GloVe ve torchtext koddan tamamen kazındı!")

# Ve motoru tekrar, bu kez gereksiz ağırlıklardan kurtulmuş olarak ateşliyoruz:
!export WANDB_MODE=offline && python main.py

In [ ]:
%cd /kaggle/working/MMTD

# BÜTÜN .py dosyalarının içine gir ve 'torchtext' veya 'GloVe' geçen her satırı acımadan sil!
!sed -i '/torchtext/d' *.py
!sed -i '/GloVe/d' *.py

print("✅ Tüm projedeki .py dosyalarından GloVe ve torchtext tamamen temizlendi!")

# Ve motoru yeniden ateşle!
!export WANDB_MODE=offline && python main.py

In [ ]:
%cd /kaggle/working/MMTD

# main.py içindeki o eski uzun parametre adını, yenisiyle değiştiriyoruz
!sed -i 's/evaluation_strategy/eval_strategy/g' main.py

print("✅ HuggingFace parametresi güncellendi!")

# Motoru (umarım bu kez gerçekten son kez) ateşliyoruz!
!export WANDB_MODE=offline && python main.py

In [ ]:
%cd /kaggle/working/MMTD

# main.py içindeki o eski ve artık desteklenmeyen komut satırını tamamen siliyoruz
!sed -i '/overwrite_output_dir/d' main.py

print("✅ HuggingFace'in son pürüzü de temizlendi!")

# Ve motoru (biliyorum bunu çok söyledik ama bu sefer gerçekten) ateşliyoruz!
!export WANDB_MODE=offline && python main.py

In [ ]:
%cd /kaggle/working/MMTD

# 1. Boş BERT modeli yerine Gerçek Çok Dilli (Multilingual) BERT'i bağlıyoruz
!sed -i 's/bert_checkpoint = None/bert_checkpoint = "bert-base-multilingual-cased"/g' main.py

# 2. Boş Resim modeli yerine Gerçek Doküman (Document/BEiT) modelini bağlıyoruz
!sed -i 's/dit_checkpoint = None/dit_checkpoint = "microsoft\/dit-base"/g' main.py

print("✅ Model bağlantıları gerçek ağırlıklarla güncellendi!")

In [ ]:
%cd /kaggle/working/MMTD
!sed -i 's/bert_checkpoint = None/bert_checkpoint = "bert-base-multilingual-cased"/g' main.py
!sed -i 's/dit_checkpoint = None/dit_checkpoint = "microsoft\/dit-base"/g' main.py

In [ ]:
!export WANDB_MODE=offline && python main.py

In [ ]:
%cd /kaggle/working/MMTD
%cat Email_dataset.py


In [ ]:
import sys

file_path = "/kaggle/working/MMTD/Email_dataset.py"

with open(file_path, "r") as file:
    lines = file.readlines()

new_lines = []

for line in lines:
    if "pic = Image.open(pic_path)" in line:
        # Eski patlayan satır yerine güvenli try-except bloğunu yerleştiriyoruz
        new_lines.append("        try:\n")
        new_lines.append("            pic = Image.open(pic_path).convert('RGB')\n")
        new_lines.append("        except FileNotFoundError:\n")
        new_lines.append("            # Bozuk/Bulunamayan dosyalar için sistemi çökertmek yerine boş bir resim ver\n")
        new_lines.append("            pic = Image.new('RGB', (224, 224), (255, 255, 255))\n")
        new_lines.append("        except Exception as e:\n")
        new_lines.append("            # Başka bir okuma hatası olursa (Bozuk jpeg vb.) yine boş resim ver\n")
        new_lines.append("            pic = Image.new('RGB', (224, 224), (255, 255, 255))\n")
    else:
        new_lines.append(line)

with open(file_path, "w") as file:
    file.writelines(new_lines)

print("✅ Ameliyat başarılı! Veri yükleyici artık Asya karakterlerine veya kayıp dosyalara karşı bağışık.")

In [ ]:
%cd /kaggle/working/MMTD
!export WANDB_MODE=offline && python main.py

In [ ]:
%cd /kaggle/working/MMTD

# 1. Eksik olan 'results' klasörünü zorla (force) oluşturuyoruz
!mkdir -p ./output/PhishGuard/results/

print("✅ Eksik klasör oluşturuldu! Şimdi son testi tekrar yapıyoruz...")

# 2. Sadece testi (evaluate) tekrar koşturmak için main.py'yi ateşle
# (Eğitim bittiği için sistem var olan checkpoint'i görüp direkt teste geçecek)
!export WANDB_MODE=offline && python main.py

In [ ]:
# 1. Ana çıktı klasörüne git
%cd /kaggle/working/MMTD/output/PhishGuard

# 2. Her şeyi (Ağırlıklar, Grafikler, Kayıtlar) paketle
!tar -cvzf /kaggle/working/PhishGuard_FINAL_SUCCESS.tar.gz .

print("🏁 HER ŞEY HAZIR! Sol/Sağ panelden PhishGuard_FINAL_SUCCESS.tar.gz dosyasını hemen indir!")

In [ ]:
%cd /kaggle/working/
# Büyük dosyayı 500'er MB'lık parçalara böl (isimleri: PhishGuard_Part_aa, ab, ac... olacak)
!split -b 500m PhishGuard_FINAL_SUCCESS.tar.gz PhishGuard_Part_

print("✅ Dosya parçalara ayrıldı! Şimdi sağ panelden 'PhishGuard_Part_aa', 'ab', 'ac' ve 'ad' dosyalarını tek tek indir.")

In [ ]:
# Yer açmak için devasa resim klasörünü siliyoruz (Modelin güvende!)
!rm -rf /kaggle/working/MMTD/DATA/email_data/pics
print("🧹 Disk temizlendi, şimdi indirme daha rahat olmalı.")

In [ ]:
from IPython.display import FileLink
# İlk parçayı (aa) indirmek için link oluştur
print("Lütfen aşağıdaki linke sağ tıklayıp 'Bağlantıyı Farklı Kaydet' de:")
FileLink(r'PhishGuard_Part_aa')

In [ ]:
from IPython.display import FileLink
import os

# Dosya yolunu tam olarak belirleyelim
file_path = '/kaggle/working/PhishGuard_FINAL_SUCCESS.tar.gz'

if os.path.exists(file_path):
    print("✅ Devasa paket bulundu! Aşağıdaki linke sağ tıklayıp 'Farklı Kaydet' de:")
    display(FileLink(file_path))
else:
    print("❌ Dosya bulunamadı. Klasör yolunu kontrol etmelisin.")

In [ ]:
from IPython.display import FileLink, display
import os

# Parçaların olduğu klasör
folder_path = '/kaggle/working/'
# Parça isimlerini listele (aa, ab, ac, ad...)
parts = [f for f in os.listdir(folder_path) if f.startswith('PhishGuard_Part_')]
parts.sort() # Sıralı gelsinler (aa, ab, ac...)

print("🚀 PARÇALI İNDİRME MERKEZİ")
print("-----------------------------------------")
for part in parts:
    print(f"📦 {part} dosyasını indirmek için tıklayın:")
    display(FileLink(part))
    print("-----------------------------------------")